# 08 — Fine-tune Jais-6.7B with QLoRA

The fourth model: a modern Arabic-native decoder-only LLM, adapted with 4-bit QLoRA. Full
fine-tuning of a 6.7B model does not fit on a 16 GB card; QLoRA does.

**Kept identical to `07_finetune_seq2seq.ipynb`** so the four-model comparison holds: the same
canonical split, the same held-out test set, the same effective batch size of 16, the same four
ROUGE protocols, the same Arabic sanity gate, the same `results.json` schema, `TEST_MODE`, and
resume support.

**Necessarily different, and documented rather than hidden:**

| | seq2seq models | Jais QLoRA |
|---|---|---|
| head | `AutoModelForSeq2SeqLM` | `AutoModelForCausalLM` |
| weights | full fp16/fp32 | frozen 4-bit NF4 + trainable LoRA adapters |
| input | encoder/decoder pair | one instruction-formatted sequence |
| loss | over all target tokens | over summary tokens only (prompt masked to `-100`) |
| decoding | beam search, `num_beams=4` | greedy, `num_beams=1` |
| saved artefact | full model | LoRA adapter (tens of MB) |
| epochs | 8 | 3 |

The last two rows are real confounds and belong in the paper's limitations, not buried here.
Beam search over a 6.7B model is prohibitively slow on a T4, and instruction-tuning a 6.7B model
converges far faster than training a small seq2seq model, so 8 epochs would be wasteful rather
than fair. State both explicitly when reporting.

**GPU: choose T4.** 4-bit NF4 runs on Pascal (P100) too, but QLoRA computes in fp16 and only the
T4 has tensor cores.

## Step 0 — Setup

In [ ]:
!nvidia-smi


`nvidia-smi` queries the driver directly, so it is ground truth for how many GPUs this session
has and whether anything is already holding memory on GPU 0. If GPU 0 already shows several GiB
in use, restart the kernel before continuing.

### A note on the `transformers` version

Jais loads custom modeling code from the Hub via `trust_remote_code=True`, and that code was
written against `transformers` 4.x. Custom remote code is the most version-fragile part of the
stack -- AraBART already needed a 4.x pin on this Kaggle image (its `BarthezTokenizer` was
rewritten in 5.x and broke). **Try the default image first.** If Step 3 fails with an
`AttributeError`, `TypeError`, or an import error from inside the downloaded Jais code, uncomment
the pin below, restart the kernel, and re-run.

In [ ]:
!pip install -q -U bitsandbytes peft accelerate evaluate rouge_score bert_score absl-py nltk qalsadi
# If Step 3 fails inside Jais's custom remote code, uncomment, restart the kernel, and re-run:
# !pip install -q "transformers==4.47.1"


In [ ]:
import os

print(f"CUDA_VISIBLE_DEVICES before override: {os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>')}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"CUDA_VISIBLE_DEVICES after override:  {os.environ['CUDA_VISIBLE_DEVICES']}")

import re, json, time, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
from nltk.stem import SnowballStemmer
from rouge_score import rouge_scorer
import qalsadi.lemmatizer
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from tqdm import tqdm

nltk.download("stopwords", quiet=True)
tqdm.pandas()

MODEL_KEY  = "jais-6p7b-qlora"
MODEL_NAME = "inceptionai/jais-family-6p7b-chat"
# Drop-in alternative if Jais is unavailable: "FreedomIntelligence/AceGPT-7B-chat".
# Step 5 detects LoRA target modules from the loaded model, so a Llama-based swap needs no edits.

MAX_ARTICLE_TOKENS  = 512     # article truncated before templating, matching the seq2seq cap
MAX_SEQ_LEN         = 768     # prompt + summary during training
GEN_MAX_NEW_TOKENS  = 150     # matches GEN_MAX_LEN elsewhere (summary tokens only)
NUM_BEAMS           = 1       # greedy -- beam search over 6.7B is impractical on a T4

LEARNING_RATE    = 2e-4       # standard QLoRA range; higher than full fine-tuning because only
                              # the small adapter matrices are updated
NUM_EPOCHS       = 3
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8          # 2 x 8 = 16 effective, identical to every other model
EVAL_BATCH_SIZE  = 2
BATCH_SIZE_GEN   = 4

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
RANDOM_SEED = 42

TEST_MODE   = False           # True -> skip training, evaluate a subset, fast end-to-end check
TEST_MODE_N = 20
TRAIN_SUBSAMPLE = None        # e.g. 8000 to cap training set size if GPU time is short

DATA_DIR = "/kaggle/input/datasets/mohamedbentalb/arabic-summ-canonical-v1/canonical_dataset"
RESUME_FROM_CHECKPOINT = None
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

assert TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS == 16, "effective batch size must match the other models"

print(f"\n=== {MODEL_KEY} ===")
print(f"  {MODEL_NAME}")
print(f"  lr={LEARNING_RATE}  epochs={NUM_EPOCHS}  LoRA r={LORA_R} alpha={LORA_ALPHA}")
print(f"  batch {TRAIN_BATCH_SIZE} x accum {GRAD_ACCUM_STEPS} = 16 effective  TEST_MODE={TEST_MODE}")

if torch.cuda.is_available():
    _gpu, _cc = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    print(f"\nGPU: {_gpu} (compute capability {_cc[0]}.{_cc[1]})")
    if _cc[0] < 6:
        print("ERROR: bitsandbytes NF4 needs compute capability >= 6.0. This GPU cannot run QLoRA.")
    elif _cc[0] < 7:
        print("ADVISORY: NF4 works here, but QLoRA computes in fp16 and this GPU has no tensor "
              "cores. A T4 will be substantially faster. Consider switching the accelerator.")
    else:
        print("GPU is suitable: NF4 supported and tensor cores available for fp16 compute.")

if not os.path.exists(f"{DATA_DIR}/manifest.json"):
    print(f"\nWARNING: no manifest.json at {DATA_DIR}. Searching /kaggle/input ...")
    for dirpath, _, filenames in os.walk("/kaggle/input"):
        if "manifest.json" in filenames:
            print(f"  Found: {dirpath}")


## Step 1 — Load the canonical split

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")

with open(f"{DATA_DIR}/manifest.json") as f:
    data_manifest = json.load(f)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"Canonical dataset built at: {data_manifest['created_at_utc']}")

assert set(train_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "train/test overlap!"
assert set(val_df["article_hash"]).isdisjoint(set(test_df["article_hash"])), "val/test overlap!"
print("Leakage re-check passed.")

if TRAIN_SUBSAMPLE is not None:
    train_df = train_df.sample(n=min(TRAIN_SUBSAMPLE, len(train_df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Subsampled training set to {len(train_df)} rows (TRAIN_SUBSAMPLE).")


## Step 2 — GPU memory check

If a previous cell crashed, its tensors stay referenced by the traceback and
`torch.cuda.empty_cache()` alone cannot reclaim them. This reports what is actually free.

In [ ]:
import gc, sys

for _v in ["model", "trainer", "optimizer"]:
    if _v in dir():
        exec(f"del {_v}")
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 0: {alloc:.2f} GiB allocated of {total:.2f} GiB")
    print("Looks clean." if alloc < 1.0 else
          "WARNING: memory already in use -- restart the kernel before continuing.")


## Step 2 — Tokenizer and prompt template

Uses the model's own chat template when it ships one, so the prompt format matches what the
model was instruction-tuned on. Falls back to an Arabic Alpaca-style template otherwise.
`padding_side="left"` is required for correct batched generation with a causal LM.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# RIGHT padding for training. Causal-LM training conventionally right-pads; the collator pads
# input_ids and labels on whichever side the tokenizer specifies, and left-padding during
# training is non-standard and can interact badly with positional handling.
# Generation is the opposite case -- batched causal-LM decoding requires LEFT padding, so
# generate_summaries() flips it temporarily and restores it afterwards.
tokenizer.padding_side = "right"

HAS_CHAT_TEMPLATE = getattr(tokenizer, "chat_template", None) is not None
print(f"Built-in chat template: {HAS_CHAT_TEMPLATE}")

INSTRUCTION = "لخص النص التالي بإيجاز ودقة بلغة عربية فصيحة"
FALLBACK_TEMPLATE = ("### التعليمات:\n" + INSTRUCTION + "\n\n### النص:\n{article}\n\n### الملخص:\n")


def build_prompt(article_text):
    truncated = tokenizer.decode(
        tokenizer(article_text, max_length=MAX_ARTICLE_TOKENS, truncation=True)["input_ids"],
        skip_special_tokens=True,
    )
    if HAS_CHAT_TEMPLATE:
        msgs = [{"role": "user", "content": f"{INSTRUCTION}:\n\n{truncated}"}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return FALLBACK_TEMPLATE.format(article=truncated)


print("\n--- example prompt ---")
print(build_prompt(test_df.iloc[0]["article"])[:500])


## Step 3 — Load the base model in 4-bit

`bnb_4bit_compute_dtype=torch.float16`, not bfloat16: a T4 is Turing and has no native bf16
support (that needs compute capability >= 8.0).

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
)
# use_gradient_checkpointing=False here because TrainingArguments enables it; setting it in
# both places double-applies the wrapper.
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)

# ...but PEFT registers the input-gradient hook *inside* that same disabled branch, so it must
# be added back explicitly. Without it, the frozen 4-bit embeddings produce activations with
# requires_grad=False; gradient checkpointing then severs the graph and backward() dies with
# "element 0 of tensors does not require grad and does not have a grad_fn".
# (PEFT's other route -- gradient_checkpointing_kwargs={"use_reentrant": False} -- does not
# help here: Jais uses the deprecated checkpointing format and silently ignores those kwargs.)
base_model.enable_input_require_grads()
print("Input-gradient hook enabled (required for gradient checkpointing + LoRA).")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Base model loaded in 4-bit ({sum(p.numel() for p in base_model.parameters())/1e9:.2f}B params).")


## Step 4 — Attach LoRA adapters

Target modules are **detected from the loaded model** rather than hardcoded. Jais uses a custom
GPT-3-style architecture whose linear layers are not named like Llama's `q_proj`/`v_proj`, so a
hardcoded list would silently attach adapters to nothing (or crash). This also means swapping in
a Llama-based model such as AceGPT needs no edits here.

In [ ]:
def find_linear_module_names(model):
    names = set()
    for name, module in model.named_modules():
        if "Linear" in type(module).__name__ and "lm_head" not in name:
            names.add(name.split(".")[-1])
    return sorted(names)


target_modules = find_linear_module_names(base_model)
print("Detected linear layers:", target_modules)
assert target_modules, "no linear layers detected -- LoRA would attach to nothing"

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
)

if RESUME_FROM_CHECKPOINT is not None:
    model = PeftModel.from_pretrained(base_model, RESUME_FROM_CHECKPOINT, is_trainable=True)
    print(f"Loaded existing adapter from {RESUME_FROM_CHECKPOINT}")
else:
    model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# LoRA silently attaching to nothing is a real failure mode with non-Llama architectures:
# training would "succeed", loss would barely move, and the model would be unchanged.
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in model.parameters())
print(f"trainable: {_trainable:,} / {_total:,} ({_trainable/_total*100:.3f}%)")
assert _trainable > 0, "no trainable parameters -- LoRA attached to nothing; check target_modules"
assert _trainable < _total * 0.10, (
    f"{_trainable/_total*100:.1f}% of parameters are trainable -- that is not LoRA. "
    "The base model was probably not frozen; check prepare_model_for_kbit_training."
)


## Step 5 — Tokenize with prompt masking

Loss is computed on the summary only. Prompt positions get `label = -100`, so the model is
supervised on generating a summary rather than on reproducing the instruction and article.

In [ ]:
def tokenize_with_masking(article, summary):
    prompt = build_prompt(article)
    full = prompt + summary + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]

    labels = list(full_ids)
    # clamp: if truncation cut inside the prompt, mask everything rather than indexing past the end
    for i in range(min(len(prompt_ids), len(full_ids))):
        labels[i] = -100
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


def tokenize_batch(examples):
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for a, s in zip(examples["article"], examples["summary"]):
        r = tokenize_with_masking(a, s)
        for k in out:
            out[k].append(r[k])
    return out


train_dataset = Dataset.from_pandas(train_df[["article", "summary"]]).map(
    tokenize_batch, batched=True, remove_columns=["article", "summary"])
eval_dataset = Dataset.from_pandas(val_df[["article", "summary"]]).map(
    tokenize_batch, batched=True, remove_columns=["article", "summary"])
print(f"train_dataset: {len(train_dataset)}  eval_dataset: {len(eval_dataset)}")

_ex = train_dataset[0]
_masked = sum(1 for l in _ex["labels"] if l == -100)
print(f"sanity: example 0 has {len(_ex['labels'])} tokens, {_masked} masked "
      f"({_masked/len(_ex['labels'])*100:.0f}% prompt), {len(_ex['labels'])-_masked} supervised")
assert _masked < len(_ex["labels"]), "every token masked -- nothing would be learned"


## Step 6 — Train

Do not interrupt. `gradient_checkpointing=True` is genuinely needed here for a 6.7B model, and
`use_cache` must be off while it is on; the cache is restored afterwards so generation stays
fast.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=None, label_pad_token_id=-100, padding=True)

training_args = TrainingArguments(
    output_dir=f"{RESULTS_DIR}/checkpoints_{MODEL_KEY}",
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    load_best_model_at_end=True,   # with PEFT this restores the best *adapter*; Step 10's
                                     # sanity probe verifies the result is a working model
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    seed=RANDOM_SEED,
    # PeftModel.forward takes **kwargs, so Trainer can fail to infer the label column and then
    # reports no eval loss -- which would silently disable early stopping and best-model
    # selection. Stating it explicitly is harmless if inference would have worked.
    label_names=["labels"],
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- Pre-flight: one forward pass must yield a loss that carries a gradient. ---
# A severed graph (missing input-grad hook, base model not frozen correctly, adapters not
# attached) otherwise only surfaces at the first backward(), after model loading and
# tokenisation have already been paid for.
_pf = data_collator([train_dataset[i] for i in range(2)])
_pf = {k: v.to(model.device) for k, v in _pf.items()}
_out = model(**_pf)
print(f"Pre-flight loss: {_out.loss.item():.4f}  requires_grad={_out.loss.requires_grad}")
assert _out.loss.requires_grad and _out.loss.grad_fn is not None, (
    "Loss has no grad_fn -- gradients cannot flow, and trainer.train() would fail with "
    "'element 0 of tensors does not require grad'. Usually enable_input_require_grads() was "
    "not called before gradient checkpointing was enabled."
)
print("Pre-flight passed: gradients can flow to the LoRA adapters.")
del _out, _pf
torch.cuda.empty_cache()

train_result = None
TRAINING_COMPLETED = False

if TEST_MODE:
    print("TEST_MODE -- skipping training.")
else:
    _cfg = getattr(model, "config", None) or model.base_model.config
    _cfg.use_cache = False   # mutually exclusive with gradient checkpointing
    if RESUME_FROM_CHECKPOINT is not None:
        print(f"Resuming from {RESUME_FROM_CHECKPOINT}")
        train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    else:
        print(f"Training {MODEL_KEY} ({NUM_EPOCHS} epochs)")
        train_result = trainer.train()
    TRAINING_COMPLETED = True
    print(train_result)

_cfg = getattr(model, "config", None) or model.base_model.config
_cfg.use_cache = True   # restore for fast generation below


## Step 7 — Training curve

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_pts = [(h.get("epoch"), h["loss"]) for h in history
             if "loss" in h and "eval_loss" not in h and h.get("epoch") is not None]
eval_pts = [(h.get("epoch"), h["eval_loss"]) for h in history
            if "eval_loss" in h and h.get("epoch") is not None]
print(f"{len(train_pts)} training points, {len(eval_pts)} validation points")

if train_pts or eval_pts:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    if train_pts:
        ax1.plot([e for e, _ in train_pts], [l for _, l in train_pts],
                 color="tab:blue", alpha=0.55, lw=1, label="Training loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Training loss", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue"); ax1.grid(alpha=0.25)
    ax2 = ax1.twinx()
    if eval_pts:
        ax2.plot([e for e, _ in eval_pts], [l for _, l in eval_pts],
                 color="tab:orange", marker="o", lw=2, label="Validation loss")
    ax2.set_ylabel("Validation loss", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
    plt.title(f"{MODEL_KEY} — training curve")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/{MODEL_KEY}_training_curve.png", dpi=150, bbox_inches="tight")
    plt.show()

for i, (ep, vl) in enumerate(eval_pts, 1):
    print(f"  Epoch {i} (epoch={ep:.2f}): {vl:.4f}")


## Step 8 — Save the LoRA adapter

`save_pretrained` on a PEFT model writes only the adapter weights — tens of MB, not the ~13 GB
base model.

In [ ]:
if TEST_MODE:
    print("TEST_MODE -- skipping save (nothing was trained).")
    adapter_path = None
else:
    adapter_path = f"{RESULTS_DIR}/{MODEL_KEY}_adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    size_mb = sum(os.path.getsize(os.path.join(adapter_path, f))
                  for f in os.listdir(adapter_path)) / 1024**2
    print(f"Saved adapter to {adapter_path} ({size_mb:.1f} MB)")


## Step 9 — Scoring protocols

One definition, four protocols. `rouge_score`'s stock tokenizer uses `NON_ALPHANUM_RE =
[^a-z0-9]+`, which reduces Arabic to an empty token list and returns exactly 0.0; its
`tokenizer=` argument also needs an object with `.tokenize()`, not a lambda. Both failures are
silent, so the self-test below asserts each protocol is non-zero on known-matching text.

In [ ]:
_PUNCT_RE = re.compile(r"([^\w\s])", flags=re.UNICODE)


def _basic_tokenize(text):
    """Lowercase, whitespace-split, split punctuation off as separate tokens (XL-Sum's BasicTokenizer)."""
    text = text.lower()
    text = _PUNCT_RE.sub(r" \1 ", text)
    return [t for t in text.split() if t]


class SplitTokenizer:
    name = "whitespace"
    def tokenize(self, text):
        return text.split()


class PunctTokenizer:
    name = "punct-aware"
    def tokenize(self, text):
        return _basic_tokenize(text)


class StemmedTokenizer:
    """Matches Kahla et al. (2023) via XL-Sum's multilingual_rouge_scoring."""
    name = "punct-aware + stemmed"
    def __init__(self):
        self.stemmer = SnowballStemmer("arabic", ignore_stopwords=True)
    def tokenize(self, text):
        return [self.stemmer.stem(t) for t in _basic_tokenize(text)]


class LemmaTokenizer:
    """Lemmatisation resolves to real dictionary entries; the stemmer truncates
    (باريس -> اريس). ~180x slower per call, so results are memoised."""
    name = "punct-aware + lemmatized"
    def __init__(self):
        self._lem = qalsadi.lemmatizer.Lemmatizer()
        self._cache = {}
    def _lemma(self, w):
        if w not in self._cache:
            try:
                self._cache[w] = self._lem.lemmatize(w) or w
            except Exception:
                self._cache[w] = w
        return self._cache[w]
    def tokenize(self, text):
        return [self._lemma(t) for t in _basic_tokenize(text)]


PROTOCOLS = [SplitTokenizer(), PunctTokenizer(), StemmedTokenizer(), LemmaTokenizer()]
ROUGE_KEYS = ["rouge1", "rouge2", "rougeL", "rougeLsum"]


def rouge_with(predictions, references, tokenizer_obj):
    scorer = rouge_scorer.RougeScorer(ROUGE_KEYS, use_stemmer=False, tokenizer=tokenizer_obj)
    totals = {k: [] for k in ROUGE_KEYS}
    for pred, ref in zip(predictions, references):
        s = scorer.score(ref, pred)
        for k in ROUGE_KEYS:
            totals[k].append(s[k].fmeasure)
    return {k: float(np.mean(v)) for k, v in totals.items()}


def compute_bertscore(predictions, references, lang="ar"):
    bs = evaluate.load("bertscore")
    r = bs.compute(predictions=predictions, references=references, lang=lang)
    return r  # per-example lists, so slices can be averaged without recomputing


_p = "أعلن متحدث باسم فرق الإطفاء أنّ الحريق الضخم اندلع في كاتدرائية نوتردام"
_r = "أعلنت فرق الإطفاء أنّ الحريق الضخم الذي اندلع في كاتدرائية نوتردام في باريس"
print("Self-test (all must be > 0):")
for _t in PROTOCOLS:
    _v = rouge_with([_p], [_r], _t)["rouge1"]
    print(f"  {_t.name:<26} rouge1 = {_v:.4f}")
    assert _v > 0, f"{_t.name} returned zero on matching Arabic -- tokenizer broken"


## Step 10 — Evaluate on the test set

Same test examples and same four scoring protocols as every other model. Generation differs:
a causal LM returns prompt + continuation, so the prompt tokens are sliced off before decoding.

In [ ]:
test_articles = test_df["article"].tolist()
test_references = test_df["summary"].tolist()
source_vals = test_df["source"].tolist() if "source" in test_df.columns else ["all"] * len(test_df)


def generate_summaries(model, tokenizer, articles, batch_size=BATCH_SIZE_GEN):
    """Batched causal-LM generation.

    Requires LEFT padding: with right padding the prompt would be followed by pad tokens, so
    the model would continue from padding rather than from the end of the prompt, and the
    prompt-slice below would cut the wrong position. Training uses right padding, so the side
    is flipped here and restored afterwards.
    """
    _prev_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    model.eval()
    preds = []
    try:
        for i in tqdm(range(0, len(articles), batch_size), desc="Generating"):
            prompts = [build_prompt(a) for a in articles[i:i + batch_size]]
            inputs = tokenizer(prompts, return_tensors="pt", truncation=True,
                               max_length=MAX_SEQ_LEN, padding=True).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS,
                                     num_beams=NUM_BEAMS, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id)
            # Left padding makes every row's prompt end at the same index, so this slice is
            # valid for the whole batch.
            new_tokens = out[:, inputs["input_ids"].shape[1]:]
            preds.extend(t.strip() for t in tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    finally:
        tokenizer.padding_side = _prev_side
    return preds


def arabic_ratio(text):
    if not text or not text.strip():
        return 0.0
    return sum(1 for ch in text if "\u0600" <= ch <= "\u06FF") / len(text)


_probe = generate_summaries(model, tokenizer, test_articles[:3])
_ratios = [arabic_ratio(p) for p in _probe]
print("Sanity probe -- Arabic character ratio:")
for _i, (_p, _r) in enumerate(zip(_probe, _ratios)):
    print(f"  [{_i}] ratio={_r:.2f}  {_p[:90]}")
assert np.mean(_ratios) > 0.5, (
    f"Model output is not Arabic (mean ratio {np.mean(_ratios):.2f}). Either training was "
    "interrupted, or the prompt template does not match what this model expects -- inspect "
    "the example prompt printed in Step 2."
)
print(f"Sanity probe passed (mean {np.mean(_ratios):.2f}).")

if TEST_MODE:
    test_articles = test_articles[:TEST_MODE_N]
    test_references = test_references[:TEST_MODE_N]
    source_vals = source_vals[:TEST_MODE_N]
    print(f"TEST_MODE: evaluating {len(test_articles)} examples.")

t0 = time.time()
predictions = generate_summaries(model, tokenizer, test_articles)
generation_time_s = time.time() - t0
empty = sum(1 for p in predictions if not p.strip())
print(f"Generated {len(predictions)} in {generation_time_s:.0f}s | empty: {empty}")

print("\n=== Sample predictions ===")
for i in range(3):
    print(f"\n[{i}] PRED: {predictions[i][:200]}")
    print(f"[{i}] REF : {test_references[i][:200]}")


In [ ]:
slices = {"ALL": list(range(len(predictions)))}
for src in sorted(set(source_vals)):
    slices[src] = [i for i, s in enumerate(source_vals) if s == src]

rows = []
for slice_name, idx in slices.items():
    sp = [predictions[i] for i in idx]
    sr = [test_references[i] for i in idx]
    for tok in PROTOCOLS:
        r = rouge_with(sp, sr, tok)
        rows.append({"slice": slice_name, "n": len(idx), "protocol": tok.name,
                     "ROUGE-1": r["rouge1"] * 100, "ROUGE-2": r["rouge2"] * 100,
                     "ROUGE-L": r["rougeL"] * 100})

results_df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.2f}")
print(results_df.to_string(index=False))


In [ ]:
METRIC_COLS = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
all_rows = results_df[results_df["slice"] == "ALL"].set_index("protocol")[METRIC_COLS]

print(f"{'protocol':<28}{'ROUGE-1':>10}{'ROUGE-2':>10}{'ROUGE-L':>10}")
for t in PROTOCOLS:
    r = all_rows.loc[t.name]
    print(f"{t.name:<28}{r['ROUGE-1']:>10.2f}{r['ROUGE-2']:>10.2f}{r['ROUGE-L']:>10.2f}")
print("-" * 58)
total = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["whitespace"]
lemma_vs_stem = all_rows.loc["punct-aware + lemmatized"] - all_rows.loc["punct-aware + stemmed"]
print(f"{'lemmatized - stemmed':<28}{lemma_vs_stem['ROUGE-1']:>+10.2f}{lemma_vs_stem['ROUGE-2']:>+10.2f}{lemma_vs_stem['ROUGE-L']:>+10.2f}")
print(f"{'TOTAL swing':<28}{total['ROUGE-1']:>+10.2f}{total['ROUGE-2']:>+10.2f}{total['ROUGE-L']:>+10.2f}")
rel = [((all_rows.loc["punct-aware + lemmatized"][m] / all_rows.loc["whitespace"][m] - 1) * 100
        if all_rows.loc["whitespace"][m] > 0 else float("nan")) for m in METRIC_COLS]
print(f"{'TOTAL relative':<28}{rel[0]:>+9.1f}%{rel[1]:>+9.1f}%{rel[2]:>+9.1f}%")


In [ ]:
bs_raw = compute_bertscore(predictions, test_references)
bert_rows = [{"slice": s, "n": len(idx),
              "precision": float(np.mean([bs_raw["precision"][i] for i in idx])),
              "recall": float(np.mean([bs_raw["recall"][i] for i in idx])),
              "f1": float(np.mean([bs_raw["f1"][i] for i in idx]))}
             for s, idx in slices.items()]
bert_df = pd.DataFrame(bert_rows)
print(bert_df.to_string(index=False))


## Step 11 — Save results

In [ ]:
SUFFIX = "_TESTMODE" if TEST_MODE else ""

pd.DataFrame({"article": test_articles, "reference": test_references,
              "prediction": predictions, "source": source_vals}
             ).to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_predictions.csv", index=False)
results_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_rouge_by_protocol.csv", index=False)
bert_df.to_csv(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_bertscore.csv", index=False)

import transformers as _tf
payload = {
    "model_key": MODEL_KEY,
    "model_name": MODEL_NAME,
    "run_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": _tf.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data_manifest_created_at": data_manifest["created_at_utc"],
    "hyperparameters": {
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS,
        "max_article_tokens": MAX_ARTICLE_TOKENS, "max_seq_len": MAX_SEQ_LEN,
        "gen_max_new_tokens": GEN_MAX_NEW_TOKENS, "num_beams": NUM_BEAMS,
        "seed": RANDOM_SEED,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": target_modules,
        "quantization": "4-bit nf4, double quant, fp16 compute",
    },
    "dataset_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "training": {
        "test_mode": TEST_MODE, "training_completed": TRAINING_COMPLETED,
        "final_train_loss": (train_result.training_loss if train_result is not None else None),
        "log_history": history,
        "validation_loss_per_epoch": [l for _, l in eval_pts],
    },
    "rouge_by_protocol": results_df.to_dict(orient="records"),
    "bertscore_by_slice": bert_df.to_dict(orient="records"),
    "generation": {"time_s": generation_time_s, "empty_outputs": int(empty)},
    "comparability_caveats": [
        "greedy decoding (num_beams=1) vs beam search (num_beams=4) for the seq2seq models",
        "3 epochs vs 8 for the seq2seq models",
        "LoRA adapters on frozen 4-bit weights vs full fine-tuning",
        "instruction-template overhead reduces the effective article budget",
    ],
}

with open(f"{RESULTS_DIR}/{MODEL_KEY}{SUFFIX}_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Saved to {RESULTS_DIR}/:")
for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {fn}")


## Notes

**Run `TEST_MODE = True` first.** With `RESUME_FROM_CHECKPOINT = None` that evaluates the
*untrained* Jais model — which is a useful zero-shot baseline in its own right, and confirms the
prompt template produces sensible Arabic before you commit GPU time.

**If it is too slow,** set `TRAIN_SUBSAMPLE = 8000`. QLoRA on 6.7B is far slower per step than
the seq2seq models. Record the subsample size in the paper if you use it.

**The `comparability_caveats` field** in `results.json` carries the genuine confounds forward so
they are not lost when the results are aggregated.